# EDA — Santiago Weather Forecast
Análisis exploratorio completo del dataset meteorológico.

**Secciones:**
1. Setup y carga de datos
2. Distribución de precipitación
3. Estacionalidad (mensual y anual)
4. Correlaciones entre variables
5. Outliers y eventos extremos
6. Días secos vs lluviosos
7. Autocorrelación (ACF/PACF)

## 1. Setup y carga de datos

In [0]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/Workspace/Repos/desareca/santiago-weather-forecast')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
import warnings
warnings.filterwarnings('ignore')

from src.data.ingestion import load_from_delta_table
from src.utils.config import *

# Estilo global
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('✅ Setup completo')

In [0]:
# Cargar datos crudos
df_raw = load_from_delta_table('weather_raw', spark)

# Renombrar columnas a nombres cortos
rename_map = {
    'precipitation_sum':          'precip',
    'temperature_2m_max':         'temp_max',
    'temperature_2m_min':         'temp_min',
    'temperature_2m_mean':        'temp_mean',
    'windspeed_10m_max':          'viento_max',
    'windgusts_10m_max':          'rafagas_max',
    'winddirection_10m_dominant': 'viento_dir',
    'precipitation_hours':        'precip_horas',
    'shortwave_radiation_sum':    'radiacion',
    'et0_fao_evapotranspiration': 'evapotransp',
    'weathercode':                'weather_code',
}
# Solo renombrar las que existan
rename_map = {k: v for k, v in rename_map.items() if k in df_raw.columns}
df = df_raw.rename(columns=rename_map)

# Ordenar y setear índice
df = df.sort_values('fecha').set_index('fecha')
df.index = pd.to_datetime(df.index)

# Solo usar split de entrenamiento para el EDA (sin data leakage)
n_train = int(len(df) * TRAIN_SPLIT)
df = df.iloc[:n_train].copy()

# Features de calendario
df['anio']    = df.index.year
df['mes']     = df.index.month
df['dia_año'] = df.index.dayofyear
df['estacion'] = df['mes'].map({
    12: 'Verano', 1: 'Verano',  2: 'Verano',
    3:  'Otoño',  4: 'Otoño',   5: 'Otoño',
    6:  'Invierno', 7: 'Invierno', 8: 'Invierno',
    9:  'Primavera', 10: 'Primavera', 11: 'Primavera'
})
df['llueve'] = (df['precip'] > 1.0).astype(int)  # umbral 1mm

print(f'📊 Registros (train): {len(df)}')
print(f'📅 Período: {df.index.min().date()} → {df.index.max().date()}')
print(f'📋 Columnas: {list(df.columns)}')
display(df.describe().round(2))

## 2. Distribución de precipitación

In [0]:
precip = df['precip']
precip_lluvia = precip[precip > 1.0]  # solo días con lluvia real

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribución de Precipitación Diaria', fontsize=15, fontweight='bold')

# Panel 1: histograma completo
axes[0].hist(precip, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Todos los días (incluye ceros)')
axes[0].set_xlabel('Precipitación (mm)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(precip.mean(), color='red', linestyle='--', label=f'Media: {precip.mean():.2f} mm')
axes[0].axvline(precip.median(), color='orange', linestyle='--', label=f'Mediana: {precip.median():.2f} mm')
axes[0].legend()

# Panel 2: solo días con lluvia > 1mm, escala log
axes[1].hist(precip_lluvia, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].set_yscale('log')
axes[1].set_title('Días con lluvia > 1mm (escala log)')
axes[1].set_xlabel('Precipitación (mm)')
axes[1].set_ylabel('Frecuencia (log)')
axes[1].axvline(precip_lluvia.mean(), color='red', linestyle='--', label=f'Media: {precip_lluvia.mean():.1f} mm')
axes[1].legend()

# Panel 3: boxplot por estación
orden_estaciones = ['Verano', 'Otoño', 'Invierno', 'Primavera']
data_box = [df[df['estacion'] == e]['precip'].values for e in orden_estaciones]
bp = axes[2].boxplot(data_box, labels=orden_estaciones, patch_artist=True, showfliers=False)
colores = ['#f4a261', '#e76f51', '#457b9d', '#52b788']
for patch, color in zip(bp['boxes'], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[2].set_title('Distribución por estación')
axes[2].set_ylabel('Precipitación (mm)')

plt.tight_layout()
plt.show()

# Stats clave
print('\n📊 Estadísticas clave:')
print(f'  Total días:          {len(precip)}')
print(f'  Días secos (≤1mm):   {(precip <= 1.0).sum()} ({(precip <= 1.0).mean()*100:.1f}%)')
print(f'  Días con lluvia:     {(precip > 1.0).sum()} ({(precip > 1.0).mean()*100:.1f}%)')
print(f'  Días con >20mm:      {(precip > 20).sum()} ({(precip > 20).mean()*100:.1f}%)')
print(f'  Máximo histórico:    {precip.max():.1f} mm  ({precip.idxmax().date()})')
print(f'  Percentil 95:        {precip.quantile(0.95):.1f} mm')
print(f'  Percentil 99:        {precip.quantile(0.99):.1f} mm')

## 3. Estacionalidad

In [0]:
meses_nombre = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

precip_mensual = df.groupby('mes')['precip'].agg(['mean', 'sum', 'std']).reset_index()
lluvia_mensual = df.groupby('mes')['llueve'].mean().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Estacionalidad Mensual de Precipitación — Santiago', fontsize=15, fontweight='bold')

# Panel 1: precipitación media por mes
bars = axes[0, 0].bar(meses_nombre, precip_mensual['mean'],
                       color=['#457b9d' if m in [6,7,8] else '#a8dadc' for m in range(1,13)],
                       edgecolor='white', alpha=0.9)
axes[0, 0].set_title('Precipitación media diaria por mes (mm)')
axes[0, 0].set_ylabel('mm/día')
axes[0, 0].bar_label(bars, fmt='%.1f', padding=2, fontsize=9)

# Panel 2: probabilidad de lluvia por mes
bars2 = axes[0, 1].bar(meses_nombre, lluvia_mensual['llueve'] * 100,
                        color=['#457b9d' if m in [6,7,8] else '#a8dadc' for m in range(1,13)],
                        edgecolor='white', alpha=0.9)
axes[0, 1].set_title('Probabilidad de lluvia por mes (%)')
axes[0, 1].set_ylabel('% días con lluvia > 1mm')
axes[0, 1].bar_label(bars2, fmt='%.1f%%', padding=2, fontsize=9)

# Panel 3: serie temporal anual (media móvil 30 días)
precip_rolling = df['precip'].rolling(30, center=True).mean()
axes[1, 0].fill_between(df.index, 0, df['precip'].values, alpha=0.2, color='steelblue', label='Diario')
axes[1, 0].plot(df.index, precip_rolling, color='#e63946', linewidth=2, label='Media móvil 30d')
axes[1, 0].set_title('Serie temporal con media móvil 30 días')
axes[1, 0].set_ylabel('Precipitación (mm)')
axes[1, 0].legend()

# Panel 4: heatmap mes x año
pivot = df.groupby(['anio', 'mes'])['precip'].sum().unstack()
pivot.columns = meses_nombre
sns.heatmap(pivot, ax=axes[1, 1], cmap='YlGnBu', linewidths=0.5,
            annot=True, fmt='.0f', cbar_kws={'label': 'mm totales'})
axes[1, 1].set_title('Precipitación total mensual por año (mm)')
axes[1, 1].set_xlabel('')

plt.tight_layout()
plt.show()

In [0]:
precip_anual = df.groupby('anio')['precip'].agg(['sum', 'mean', lambda x: (x > 1.0).sum()]).reset_index()
precip_anual.columns = ['anio', 'total_mm', 'media_dia', 'dias_lluvia']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Variabilidad Anual de Precipitación — Santiago', fontsize=15, fontweight='bold')

# Panel 1: total anual
bars = axes[0].bar(precip_anual['anio'].astype(str), precip_anual['total_mm'],
                    color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axhline(precip_anual['total_mm'].mean(), color='red', linestyle='--',
                label=f"Media: {precip_anual['total_mm'].mean():.0f} mm")
axes[0].set_title('Precipitación total anual (mm)')
axes[0].set_ylabel('mm')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()
axes[0].bar_label(bars, fmt='%.0f', padding=2, fontsize=9)

# Panel 2: días con lluvia por año
bars2 = axes[1].bar(precip_anual['anio'].astype(str), precip_anual['dias_lluvia'],
                     color='#52b788', edgecolor='white', alpha=0.85)
axes[1].axhline(precip_anual['dias_lluvia'].mean(), color='red', linestyle='--',
                label=f"Media: {precip_anual['dias_lluvia'].mean():.0f} días")
axes[1].set_title('Días con lluvia por año (>1mm)')
axes[1].set_ylabel('Días')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
axes[1].bar_label(bars2, fmt='%.0f', padding=2, fontsize=9)

# Panel 3: día del año promedio (estacionalidad media histórica)
precip_dia = df.groupby('dia_año')['precip'].mean().rolling(15, center=True).mean()
axes[2].fill_between(precip_dia.index, 0, precip_dia.values, color='steelblue', alpha=0.7)
for mes_inicio, nombre in zip([1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335],
                                meses_nombre):
    axes[2].axvline(mes_inicio, color='gray', linestyle=':', alpha=0.5, linewidth=0.8)
    axes[2].text(mes_inicio + 5, precip_dia.max() * 0.95, nombre, fontsize=8, color='gray')
axes[2].set_title('Precipitación media por día del año (suavizado 15d)')
axes[2].set_xlabel('Día del año')
axes[2].set_ylabel('mm')

plt.tight_layout()
plt.show()

print('\n📊 Resumen anual:')
display(precip_anual)

## 4. Correlaciones entre variables y precipitación

In [0]:
# Variables numéricas disponibles
vars_modelo = [c for c in df.columns if c not in
               ['anio', 'mes', 'dia_año', 'estacion', 'llueve', 'viento_dir']]

corr_matrix = df[vars_modelo].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, ax=ax,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 9},
    linewidths=0.5, square=True,
    cbar_kws={'shrink': 0.8, 'label': 'Correlación de Pearson'}
)
ax.set_title('Matriz de Correlaciones — Variables Meteorológicas', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Top correlaciones con precipitación
print('\n🎯 Correlaciones con precipitación (ordenadas):')
corr_precip = corr_matrix['precip'].drop('precip').sort_values(key=abs, ascending=False)
for var, val in corr_precip.items():
    signo = '🔴' if val < -0.3 else ('🟢' if val > 0.3 else '⚪')
    print(f'  {signo} {var:30s}: {val:+.3f}')

In [0]:
# Top 6 variables más correlacionadas con precipitación
top_vars = corr_precip.abs().head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Variables más correlacionadas con precipitación', fontsize=14, fontweight='bold')
axes = axes.flatten()

colores_estacion = {'Verano': '#f4a261', 'Otoño': '#e76f51', 'Invierno': '#457b9d', 'Primavera': '#52b788'}

for i, var in enumerate(top_vars):
    for estacion, color in colores_estacion.items():
        mask = df['estacion'] == estacion
        axes[i].scatter(
            df.loc[mask, var],
            df.loc[mask, 'precip'],
            alpha=0.3, s=10, color=color, label=estacion
        )
    r = corr_matrix.loc[var, 'precip']
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Precipitación (mm)')
    axes[i].set_title(f'{var}  (r = {r:+.3f})')
    axes[i].set_ylim(-1, df['precip'].quantile(0.995))
    if i == 0:
        axes[i].legend(fontsize=8, markerscale=2)

plt.tight_layout()
plt.show()

## 5. Outliers y eventos extremos

In [0]:
# Umbrales de eventos
p90 = df['precip'].quantile(0.90)
p95 = df['precip'].quantile(0.95)
p99 = df['precip'].quantile(0.99)

extremos = df[df['precip'] >= p95][['precip', 'temp_max', 'temp_min', 'viento_max',
                                     'radiacion', 'evapotransp', 'estacion']].copy()
extremos = extremos.sort_values('precip', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Eventos Extremos de Precipitación', fontsize=14, fontweight='bold')

# Panel 1: serie temporal con eventos marcados
axes[0].fill_between(df.index, 0, df['precip'], alpha=0.3, color='steelblue')
axes[0].plot(df.index, df['precip'], linewidth=0.5, color='steelblue', alpha=0.6)
mask_extremo = df['precip'] >= p95
axes[0].scatter(df.index[mask_extremo], df['precip'][mask_extremo],
                color='red', s=20, zorder=5, label=f'Eventos ≥ p95 ({p95:.1f}mm)')
axes[0].axhline(p90, color='orange', linestyle='--', alpha=0.7, label=f'p90: {p90:.1f}mm')
axes[0].axhline(p95, color='red',    linestyle='--', alpha=0.7, label=f'p95: {p95:.1f}mm')
axes[0].axhline(p99, color='darkred',linestyle='--', alpha=0.7, label=f'p99: {p99:.1f}mm')
axes[0].set_title('Serie temporal con eventos extremos marcados')
axes[0].set_ylabel('Precipitación (mm)')
axes[0].legend(fontsize=9)

# Panel 2: eventos extremos por mes
extremos_mes = df[df['precip'] >= p95].groupby('mes').size()
extremos_mes = extremos_mes.reindex(range(1, 13), fill_value=0)
axes[1].bar(meses_nombre, extremos_mes.values, color='#e63946', edgecolor='white', alpha=0.85)
axes[1].set_title(f'Eventos extremos por mes (≥ p95 = {p95:.1f}mm)')
axes[1].set_ylabel('Cantidad de eventos')

plt.tight_layout()
plt.show()

print(f'\n🌩️  Top 15 eventos extremos (≥ p95 = {p95:.1f}mm):')
display(extremos.head(15))

print(f'\n📊 Resumen por estación:')
display(df[df['precip'] >= p95].groupby('estacion')['precip']
        .agg(['count', 'mean', 'max'])
        .rename(columns={'count': 'n_eventos', 'mean': 'media_mm', 'max': 'max_mm'})
        .round(1))

## 6. Días secos vs lluviosos

In [0]:
seco    = df[df['llueve'] == 0]
lluvioso = df[df['llueve'] == 1]

vars_comparar = [c for c in ['temp_max', 'temp_min', 'temp_mean', 'viento_max',
                              'rafagas_max', 'radiacion', 'evapotransp', 'precip_horas']
                 if c in df.columns]

n_vars = len(vars_comparar)
ncols = 4
nrows = (n_vars + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5 * nrows))
fig.suptitle('Distribución de variables: Días Secos vs Lluviosos', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, var in enumerate(vars_comparar):
    axes[i].hist(seco[var].dropna(), bins=40, alpha=0.6, color='#f4a261',
                 density=True, label=f'Seco (n={len(seco)})')
    axes[i].hist(lluvioso[var].dropna(), bins=40, alpha=0.6, color='#457b9d',
                 density=True, label=f'Lluvioso (n={len(lluvioso)})')
    axes[i].set_title(var)
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Densidad')

# Ocultar ejes vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

# Tabla comparativa
print('\n📊 Medias por condición (seco vs lluvioso):')
comparativa = pd.DataFrame({
    'Seco':     seco[vars_comparar].mean(),
    'Lluvioso': lluvioso[vars_comparar].mean(),
}).round(2)
comparativa['Diferencia'] = (comparativa['Lluvioso'] - comparativa['Seco']).round(2)
comparativa['Diferencia_%'] = ((comparativa['Diferencia'] / comparativa['Seco'].abs()) * 100).round(1)
display(comparativa)

In [0]:
# Analizar rachas (streaks)
def calcular_rachas(serie_binaria):
    rachas = []
    contador = 1
    valor_actual = serie_binaria.iloc[0]
    for val in serie_binaria.iloc[1:]:
        if val == valor_actual:
            contador += 1
        else:
            rachas.append({'tipo': 'seco' if valor_actual == 0 else 'lluvia', 'duracion': contador})
            contador = 1
            valor_actual = val
    rachas.append({'tipo': 'seco' if valor_actual == 0 else 'lluvia', 'duracion': contador})
    return pd.DataFrame(rachas)

rachas_df = calcular_rachas(df['llueve'])
rachas_seco   = rachas_df[rachas_df['tipo'] == 'seco']['duracion']
rachas_lluvia = rachas_df[rachas_df['tipo'] == 'lluvia']['duracion']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Rachas de Días Secos y Lluviosos', fontsize=14, fontweight='bold')

axes[0].hist(rachas_seco, bins=40, color='#f4a261', edgecolor='white', alpha=0.85)
axes[0].set_title(f'Rachas secas — media: {rachas_seco.mean():.1f}d, max: {rachas_seco.max()}d')
axes[0].set_xlabel('Duración (días)')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(rachas_lluvia, bins=20, color='#457b9d', edgecolor='white', alpha=0.85)
axes[1].set_title(f'Rachas lluviosas — media: {rachas_lluvia.mean():.1f}d, max: {rachas_lluvia.max()}d')
axes[1].set_xlabel('Duración (días)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

print(f'\n📊 Estadísticas de rachas:')
print(f'  Rachas secas   — total: {len(rachas_seco)}, media: {rachas_seco.mean():.1f}d, p90: {rachas_seco.quantile(0.9):.0f}d, max: {rachas_seco.max()}d')
print(f'  Rachas lluvia  — total: {len(rachas_lluvia)}, media: {rachas_lluvia.mean():.1f}d, p90: {rachas_lluvia.quantile(0.9):.0f}d, max: {rachas_lluvia.max()}d')

## 7. Autocorrelación (ACF/PACF)

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('Autocorrelación de Precipitación Diaria', fontsize=14, fontweight='bold')

# ACF / PACF — todos los días
plot_acf(df['precip'].fillna(0),  lags=60, ax=axes[0, 0], title='ACF — Todos los días (60 lags)', alpha=0.05)
plot_pacf(df['precip'].fillna(0), lags=60, ax=axes[0, 1], title='PACF — Todos los días (60 lags)', alpha=0.05, method='ywm')

# ACF / PACF — solo días lluviosos (serie binaria)
plot_acf(df['llueve'],  lags=60, ax=axes[1, 0], title='ACF — Indicador lluvia binario (60 lags)', alpha=0.05)
plot_pacf(df['llueve'], lags=60, ax=axes[1, 1], title='PACF — Indicador lluvia binario (60 lags)', alpha=0.05, method='ywm')

plt.tight_layout()
plt.show()

In [0]:
# ¿Cuánto predice cada variable en el día anterior sobre la precipitación de hoy?
vars_lag = [c for c in ['temp_max', 'temp_min', 'temp_mean', 'viento_max',
                         'rafagas_max', 'radiacion', 'evapotransp', 'weather_code']
            if c in df.columns]

# Correlación de Pearson: var(t-lag) vs precip(t), para lag 1..14
lags_range = range(1, 15)
corr_lag_df = pd.DataFrame(index=lags_range, columns=vars_lag, dtype=float)

for lag in lags_range:
    for var in vars_lag:
        corr_lag_df.loc[lag, var] = df['precip'].corr(df[var].shift(lag))

# También agregar lags de la propia precipitación
corr_lag_df['precip_lag'] = [df['precip'].corr(df['precip'].shift(lag)) for lag in lags_range]

fig, ax = plt.subplots(figsize=(16, 6))
for col in corr_lag_df.columns:
    ls = '-' if col != 'precip_lag' else '--'
    lw = 1.5 if col != 'precip_lag' else 2.5
    ax.plot(corr_lag_df.index, corr_lag_df[col], marker='o', markersize=4,
            label=col, linestyle=ls, linewidth=lw)

ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(0.1, color='gray', linestyle=':', alpha=0.5)
ax.axhline(-0.1, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Lag (días anteriores)')
ax.set_ylabel('Correlación con precipitación del día')
ax.set_title('Correlación cruzada: var(t-lag) → precip(t)', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.set_xticks(list(lags_range))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\n🎯 Correlación de cada variable en lag=1 (día anterior → precipitación hoy):')
lag1 = corr_lag_df.loc[1].sort_values(key=abs, ascending=False)
for var, val in lag1.items():
    signo = '🔴' if val < -0.15 else ('🟢' if val > 0.15 else '⚪')
    print(f'  {signo} {var:30s}: {val:+.3f}')

## Resumen de hallazgos

In [0]:
print('=' * 65)
print('RESUMEN DE HALLAZGOS EDA')
print('=' * 65)

pct_seco = (df['precip'] <= 1.0).mean() * 100
pct_lluvia = (df['precip'] > 1.0).mean() * 100
pct_extremo = (df['precip'] >= p95).mean() * 100

print(f'''
1. DISTRIBUCIÓN
   • {pct_seco:.1f}% días secos (≤1mm) — desbalanceo severo
   • {pct_lluvia:.1f}% días con lluvia real (>1mm)
   • {pct_extremo:.1f}% días con eventos extremos (≥p95={p95:.1f}mm)
   • Distribución muy asimétrica (cola derecha larga)

2. ESTACIONALIDAD
   • Patrón mediterráneo claro: lluvia concentrada en invierno (Jun-Ago)
   • Verano (Dic-Feb) prácticamente seco
   • Variabilidad interanual alta (años secos vs húmedos)

3. CORRELACIONES CON PRECIPITACIÓN
   • Variables más correlacionadas en mismo día:
     - weather_code (positivo): el tipo de evento climático predice bien
     - radiacion (negativo): menos sol → más lluvia
     - evapotransp (negativo): días húmedos tienen menos ET
   • La autocorrelación de precipitación (lag=1) es BAJA → confirma
     que los lags de precip solos no son buenos predictores

4. DÍAS SECOS VS LLUVIOSOS
   • Los días lluviosos tienen temp_max claramente menor
   • Radiación significativamente más baja en días lluviosos
   • Viento y ráfagas más altos en días lluviosos

5. IMPLICACIONES PARA EL MODELO
   • Incluir variables de lag=1 de: weather_code, radiacion,
     evapotransp, temp_max, viento_max — son los más informativos
   • Incluir estacionalidad (mes_sin/cos) — patrón muy marcado
   • Considerar rachas: días_secos_consecutivos como feature
   • El weather_code del día anterior puede ser muy útil
''')
print('=' * 65)